In [53]:
# ============================================================
# ЯЧЕЙКА 1: Импорты и исходные данные
# ============================================================
import json
import csv
import os
from collections import Counter

# Схемы таблиц
Commission = dict(id=int, name=str, code=str, specialization=int)
Specialization = dict(id=int, code=str)
SpecializationInCommission = dict(id=int, commission=int, specialization=int)
Scientist = dict(id=int, name=str, commission=int, specialization=int)
Publication = dict(id=int, title=str, scientist=int, journal=int)
Journal = dict(id=int, title=str)
JournalSpecialization = dict(journal=int, specialization=int)
ScientistInCommission = dict(scientist=int, commission=int, specialization=int)

# Таблицы данных
CommissionTable = [
    (1, "Совет по математике", "МАТ-01", 101),
    (2, "Совет по физике", "ФИЗ-02", 102),
    (3, "Совет по информатике", "ИНФ-03", 103),
]

SpecializationTable = [
    (101, "01.01.01"),
    (102, "01.04.02"),
    (103, "05.13.11"),
]

SpecializationInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

ScientistTable = [
    (1, "Иванов И.И.", 1, 101),
    (2, "Петров П.П.", 1, 102),
    (3, "Сидоров С.С.", 2, 102),
    (4, "Кузнецова А.А.", 3, 103),
]

PublicationTable = [
    (1, "Математические модели", 1, 10),
    (2, "Квантовая физика", 2, 11),
    (3, "Искусственный интеллект", 4, 12),
    (4, "Численные методы", 1, 13),
]

JournalTable = [
    (10, "Вестник РАН"),
    (11, "ЖЭТФ"),
    (12, "AI Journal"),
    (13, "Выч. математика"),
]

JournalSpecializationTable = [
    (10, 101),
    (10, 102),
    (11, 102),
    (12, 103),
    (13, 101),
]

ScientistInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

# Множество источников S
S = [
    ("Commission", Commission, CommissionTable),
    ("Specialization", Specialization, SpecializationTable),
    ("SpecializationInCommission", SpecializationInCommission, SpecializationInCommissionTable),
    ("Scientist", Scientist, ScientistTable),
    ("Publication", Publication, PublicationTable),
    ("Journal", Journal, JournalTable),
    ("JournalSpecialization", JournalSpecialization, JournalSpecializationTable),
    ("ScientistInCommission", ScientistInCommission, ScientistInCommissionTable),
]

In [54]:
# ЯЧЕЙКА 2 (исправленная v2): Генератор с полными RELATION_NAMES
import json

RELATION_NAMES = {
    # Атрибуты по сущностям
    ("name", "Учёный", None): "фио",
    ("name", "Диссовет", None): "название",
    ("name", None, None): "название",
    ("title", None, None): "название",
    ("code", None, None): "шифр",
    
    # Только прямые связи
    ("Учёный", "Публикация"): "опубликовал_статью",
    ("Публикация", "Журнал"): "опубликована_в",
    ("Журнал", "Специальность"): "покрывает_специальность",
    ("Диссовет", "Специальность"): "включает_специальность",
    ("Учёный", "Диссовет"): "состоит_в_совете",
    ("Учёный", "Специальность"): "представляет_специальность",
}

entity_resolution = {
    "mappings": [
        # Scientist
        {"source_field": "Scientist.id", "target_entity": "Учёный", "is_primary_key": True},
        {"source_field": "Scientist.name", "target_entity": None},
        {"source_field": "Scientist.commission", "target_entity": "Диссовет", "references": "Commission.id"},
        {"source_field": "Scientist.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
        
        # Publication
        {"source_field": "Publication.id", "target_entity": "Публикация", "is_primary_key": True},
        {"source_field": "Publication.title", "target_entity": None},
        {"source_field": "Publication.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
        {"source_field": "Publication.journal", "target_entity": "Журнал", "references": "Journal.id"},
        
        # Journal
        {"source_field": "Journal.id", "target_entity": "Журнал", "is_primary_key": True},
        {"source_field": "Journal.title", "target_entity": None},
        
        # JournalSpecialization (связующая)
        {"source_field": "JournalSpecialization.journal", "target_entity": "Журнал", "references": "Journal.id"},
        {"source_field": "JournalSpecialization.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
        
        # Commission
        {"source_field": "Commission.id", "target_entity": "Диссовет", "is_primary_key": True},
        {"source_field": "Commission.name", "target_entity": None},
        {"source_field": "Commission.code", "target_entity": None},
        {"source_field": "Commission.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
        
        # Specialization
        {"source_field": "Specialization.id", "target_entity": "Специальность", "is_primary_key": True},
        {"source_field": "Specialization.code", "target_entity": None},
        
        # SpecializationInCommission (связующая — убрали id из значимых)
        {"source_field": "SpecializationInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
        {"source_field": "SpecializationInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
        
        # ScientistInCommission (связующая)
        {"source_field": "ScientistInCommission.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
        {"source_field": "ScientistInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
        {"source_field": "ScientistInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
    ]
}


def get_entity_for_field(source_name, field_name):
    full_field = f"{source_name}.{field_name}"
    for mapping in entity_resolution["mappings"]:
        if mapping["source_field"] == full_field:
            return mapping["target_entity"]
    return None


def is_primary_key(source_name, field_name):
    full_field = f"{source_name}.{field_name}"
    for mapping in entity_resolution["mappings"]:
        if mapping["source_field"] == full_field:
            return mapping.get("is_primary_key", False)
    return False


def is_foreign_key(source_name, field_name):
    full_field = f"{source_name}.{field_name}"
    for mapping in entity_resolution["mappings"]:
        if mapping["source_field"] == full_field:
            return "references" in mapping
    return False

REFERENCE_TABLES = ["Publication"]

def get_relation_name(field_name, subject_entity, object_entity):
    # Связь между сущностями — только если есть в словаре
    if subject_entity and object_entity:
        key = (subject_entity, object_entity)
        if key in RELATION_NAMES:
            return RELATION_NAMES[key]
        return None  # Нет в словаре — не создаём правило
    
    # Атрибут сущности
    if subject_entity and not object_entity:
        key = (field_name, subject_entity, None)
        if key in RELATION_NAMES:
            return RELATION_NAMES[key]
    
    # Общий атрибут
    key = (field_name, None, None)
    if key in RELATION_NAMES:
        return RELATION_NAMES[key]
    
    return field_name

def generate_rules_for_table(source_name, schema):
    columns = list(schema.keys())
    rules = []
    
    has_id = "id" in columns
    id_is_primary = has_id and is_primary_key(source_name, "id")
    
    if has_id and id_is_primary:
        subject_entity = get_entity_for_field(source_name, "id")
        
        for field in columns:
            if field == "id":
                continue
            
            object_entity = get_entity_for_field(source_name, field)
            
            if is_foreign_key(source_name, field):
                if source_name in REFERENCE_TABLES:
                    # Прямая связь
                    relation1 = get_relation_name(None, object_entity, subject_entity)
                    if relation1:
                        rules.append({
                            "source": source_name,
                            "subject_field": field,
                            "subject_entity": object_entity,
                            "relation": relation1,
                            "object_field": "id",
                            "object_entity": subject_entity,
                            "time_default": "2025-01-01",
                            "confidence": 1.0
                        })
                    # Обратная связь
                    relation2 = get_relation_name(None, subject_entity, object_entity)
                    if relation2:
                        rules.append({
                            "source": source_name,
                            "subject_field": "id",
                            "subject_entity": subject_entity,
                            "relation": relation2,
                            "object_field": field,
                            "object_entity": object_entity,
                            "time_default": "2025-01-01",
                            "confidence": 1.0
                        })
            else:
                relation = get_relation_name(field, subject_entity, None)
                rules.append({
                    "source": source_name,
                    "subject_field": "id",
                    "subject_entity": subject_entity,
                    "relation": relation,
                    "object_field": field,
                    "object_entity": object_entity,
                    "time_default": "2025-01-01",
                    "confidence": 1.0
                })
    
    else:
        fk_fields = [col for col in columns if is_foreign_key(source_name, col)]
        for i, f1 in enumerate(fk_fields):
            for f2 in fk_fields[i+1:]:
                e1 = get_entity_for_field(source_name, f1)
                e2 = get_entity_for_field(source_name, f2)
                relation = get_relation_name(None, e1, e2)
                if relation:  # Только если есть в словаре
                    rules.append({
                        "source": source_name,
                        "subject_field": f1,
                        "subject_entity": e1,
                        "relation": relation,
                        "object_field": f2,
                        "object_entity": e2,
                        "time_default": "2025-01-01",
                        "confidence": 1.0
                    })
    
    return rules

def generate_all_rules(S):
    """Генерирует правила, убирая дубликаты."""
    all_rules = []
    seen = set()
    
    for name, schema, table in S:
        for rule in generate_rules_for_table(name, schema):
            # Ключ уникальности: (subject_entity, relation, object_entity)
            key = (rule.get("subject_entity"), rule["relation"], rule.get("object_entity"))
            if key not in seen:
                seen.add(key)
                all_rules.append(rule)
    
    return all_rules


generated_rules = generate_all_rules(S)

rules_json = {
    "ontology": {
        "name": "Онтология академической среды",
        "classes": ["Учёный", "Публикация", "Журнал", "Специальность", "Диссовет"],
        "relations": sorted(set(r["relation"] for r in generated_rules))
    },
    "entity_resolution": entity_resolution,
    "rules": {
        "structured": generated_rules,
        "semi_structured": [],
        "unstructured": []
    }
}

with open("ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)

print(f"✓ Сгенерировано {len(generated_rules)} правил")
print("\nПравила:")
for rule in generated_rules:
    subj = f"{rule['subject_entity']}.{rule['subject_field']}" if rule['subject_entity'] else rule['subject_field']
    obj = f"{rule['object_entity']}.{rule['object_field']}" if rule['object_entity'] else rule['object_field']
    print(f"  {subj} --{rule['relation']}--> {obj}")

✓ Сгенерировано 12 правил

Правила:
  Диссовет.id --название--> name
  Диссовет.id --шифр--> code
  Специальность.id --шифр--> code
  Диссовет.commission --включает_специальность--> Специальность.specialization
  Учёный.id --фио--> name
  Публикация.id --название--> title
  Учёный.scientist --опубликовал_статью--> Публикация.id
  Публикация.id --опубликована_в--> Журнал.journal
  Журнал.id --название--> title
  Журнал.journal --покрывает_специальность--> Специальность.specialization
  Учёный.scientist --состоит_в_совете--> Диссовет.commission
  Учёный.scientist --представляет_специальность--> Специальность.specialization


In [57]:
# ЯЧЕЙКА 3: KISS + все проверки псевдокода
def integrate(S, rules_file="ontology_rules.json"):
    """Применяет правила к данным, возвращает факты."""
    config = json.load(open(rules_file, 'r', encoding='utf-8'))
    data = {name: (sch, tbl) for name, sch, tbl in S}
    facts = []
    
    for rule in config["rules"]["structured"]:
        src = rule["source"]
        if src not in data:           
            continue
        
        schema, table = data[src]
        cols = list(schema.keys())
        
        required = [rule["subject_field"], rule["object_field"]]
        if not all(attr in schema for attr in required):
            continue
        
        for row in table:             
            d = dict(zip(cols, row))
            
            s_ent = rule.get("subject_entity")
            s_val = d[rule["subject_field"]]
            if s_val is None:       
                continue
            subj = f"{s_ent}_{s_val}" if s_ent else str(s_val)
            
            # Строка 11: объект
            if "object_value" in rule:
                obj = rule["object_value"]
            else:
                o_ent = rule.get("object_entity")
                o_val = d[rule["object_field"]]
                if o_val is None:   
                    continue
                obj = f"{o_ent}_{o_val}" if o_ent else str(o_val)
            
            t = rule.get("time_field")
            time_val = str(d[t]) if t and t in d and d[t] is not None else rule.get("time_default", "2025-01-01")
            
            v = rule.get("value_field")
            n = d[v] if v and v in d else None
            if n is not None and n < 0:
                continue
            
            facts.append((subj, rule["relation"], obj, time_val, n, rule.get("confidence", 1.0)))
    
    return facts

In [58]:
# ЯЧЕЙКА 4: Запуск с KISS-функцией
facts = integrate(S)

print(f"Всего фактов: {len(facts)}")
print(f"\nРаспределение по отношениям:")
stats = Counter(f[1] for f in facts)
for rel, count in sorted(stats.items()):
    print(f"  {rel}: {count}")

# Сохранение
import os
output_path = "experiment_data_soviets/generated/all_facts.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['subject', 'relation', 'object', 'time', 'n_value', 'confidence'])
    writer.writerows(facts)
print(f"\n✓ Сохранено в {output_path}")

Всего фактов: 46

Распределение по отношениям:
  включает_специальность: 4
  название: 11
  опубликовал_статью: 4
  опубликована_в: 4
  покрывает_специальность: 5
  представляет_специальность: 4
  состоит_в_совете: 4
  фио: 4
  шифр: 6

✓ Сохранено в experiment_data_soviets/generated/all_facts.csv


In [46]:
# ЯЧЕЙКА: Сохранение всех файлов проекта
import json
import csv
import os

# Создаём папку проекта
project_dir = "dissovet_ontology"
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/data", exist_ok=True)

# Сохраняем ячейки как Python-скрипт
with open(f"{project_dir}/module1_integrator.py", 'w', encoding='utf-8') as f:
    f.write("""# Здесь ваш код из ячейки 3 (функции integrate, apply_rule, load_rules, save_to_csv)
""")

# Сохраняем правила
with open(f"{project_dir}/ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)

# Сохраняем CSV-файлы
for name, schema, table in S:
    columns = list(schema.keys())
    with open(f"{project_dir}/data/{name}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(table)

# Сохраняем README.md
with open(f"{project_dir}/README.md", 'w', encoding='utf-8') as f:
    f.write("""# Онтологическая интеграция данных диссертационных советов

Проект выполняет интеграцию гетерогенных данных в унифицированные спецификации фактов (УСФ).

## Структура

- `ontology_rules.json` — правила отображения
- `data/*.csv` — исходные таблицы
- `module1_integrator.py` — модуль интеграции

## Запуск

Скопируйте функции из `module1_integrator.py` в ноутбук.
""")

# Сохраняем .gitignore
with open(f"{project_dir}/.gitignore", 'w') as f:
    f.write("""__pycache__/
*.pyc
.ipynb_checkpoints/
*.db
experiment_data*/
""")

print("✓ Файлы сохранены в dissovet_ontology/")

✓ Файлы сохранены в dissovet_ontology/


In [13]:
# ЯЧЕЙКА 2: Сохранение правил в JSON (с entity_resolution)
import json

rules_json = {
    "ontology": {
        "name": "Онтология академической среды",
        "classes": ["Учёный", "Публикация", "Журнал", "Специальность", "Диссовет"],
        "relations": [
            "фио", "название", "шифр", "код",
            "опубликовал_статью", "опубликована_в",
            "покрывает_специальность", "включает_специальность",
            "состоит_в_совете", "представляет_специальность"
        ]
    },

    "entity_resolution": {
        "mappings": [
            {"source_field": "Scientist.id", "target_entity": "Учёный", "is_primary_key": True},
            {"source_field": "Publication.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
            {"source_field": "Publication.id", "target_entity": "Публикация", "is_primary_key": True},
            {"source_field": "Publication.journal", "target_entity": "Журнал", "references": "Journal.id"},
            {"source_field": "Journal.id", "target_entity": "Журнал", "is_primary_key": True},
            {"source_field": "JournalSpecialization.journal", "target_entity": "Журнал", "references": "Journal.id"},
            {"source_field": "JournalSpecialization.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
            {"source_field": "Specialization.id", "target_entity": "Специальность", "is_primary_key": True},
            {"source_field": "Commission.id", "target_entity": "Диссовет", "is_primary_key": True},
            {"source_field": "SpecializationInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
            {"source_field": "SpecializationInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"},
            {"source_field": "ScientistInCommission.scientist", "target_entity": "Учёный", "references": "Scientist.id"},
            {"source_field": "ScientistInCommission.commission", "target_entity": "Диссовет", "references": "Commission.id"},
            {"source_field": "ScientistInCommission.specialization", "target_entity": "Специальность", "references": "Specialization.id"}
        ]
    },

    "rules": {
        "structured": [
            {"source": "Scientist", "subject_field": "id", "subject_entity": "Учёный", "relation": "фио", "object_field": "name", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "id", "subject_entity": "Публикация", "relation": "название", "object_field": "title", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "опубликовал_статью", "object_field": "id", "object_entity": "Публикация", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "id", "subject_entity": "Публикация", "relation": "опубликована_в", "object_field": "journal", "object_entity": "Журнал", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Journal", "subject_field": "id", "subject_entity": "Журнал", "relation": "название", "object_field": "title", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "JournalSpecialization", "subject_field": "journal", "subject_entity": "Журнал", "relation": "покрывает_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Commission", "subject_field": "id", "subject_entity": "Диссовет", "relation": "название", "object_field": "name", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Commission", "subject_field": "id", "subject_entity": "Диссовет", "relation": "шифр", "object_field": "code", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Specialization", "subject_field": "id", "subject_entity": "Специальность", "relation": "код", "object_field": "code", "object_entity": None, "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "SpecializationInCommission", "subject_field": "commission", "subject_entity": "Диссовет", "relation": "включает_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "ScientistInCommission", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "состоит_в_совете", "object_field": "commission", "object_entity": "Диссовет", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "ScientistInCommission", "subject_field": "scientist", "subject_entity": "Учёный", "relation": "представляет_специальность", "object_field": "specialization", "object_entity": "Специальность", "time_default": "2025-01-01", "confidence": 1.0}
        ],
        "semi_structured": [],
        "unstructured": []
    }
}

with open("ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)
print("✓ ontology_rules.json сохранён")

✓ ontology_rules.json сохранён


In [40]:
# ЯЧЕЙКА 3: Функции интеграции
def apply_rule(source_name, rule, data_sources):
    facts = []
    if source_name not in data_sources:
        return facts
    
    schema, table = data_sources[source_name]
    columns = list(schema.keys())
    
    for row in table:
        row_dict = dict(zip(columns, row))
        
        # Субъект
        subj_field = rule["subject_field"]
        subj_entity = rule.get("subject_entity")
        if subj_entity:
            subject = f"{subj_entity}_{row_dict[subj_field]}"
        else:
            subject = str(row_dict[subj_field])
        
        # Объект
        if "object_value" in rule:
            obj = rule["object_value"]
        else:
            obj_field = rule["object_field"]
            obj_entity = rule.get("object_entity")
            if obj_entity:
                obj = f"{obj_entity}_{row_dict[obj_field]}"
            else:
                obj = str(row_dict[obj_field])
        
        # Время
        time_val = rule.get("time_default", "2025-01-01")
        if "time_field" in rule and rule["time_field"] in row_dict:
            time_val = str(row_dict[rule["time_field"]])
        
        # Значение
        value = None
        if "value_field" in rule and rule["value_field"] in row_dict:
            value = row_dict[rule["value_field"]]
        
        facts.append((subject, rule["relation"], obj, time_val, value, rule.get("confidence", 1.0)))
    
    return facts


def integrate(S, rules_file="ontology_rules.json"):
    """Интеграция всех источников по правилам."""
    with open(rules_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    ontology = config["ontology"]
    rules = config["rules"]
    
    data_sources = {name: (schema, table) for name, schema, table in S}
    all_facts, stats = [], Counter()
    
    for rule_type in ["structured", "semi_structured", "unstructured"]:
        for rule in rules.get(rule_type, []):
            facts = apply_rule(rule["source"], rule, data_sources)
            all_facts.extend(facts)
            stats[rule["relation"]] += len(facts)
    
    return all_facts, stats, ontology

print("✓ integrate и apply_rule загружены")

✓ integrate и apply_rule загружены
